# SAM3 Zero-Shot Mask Quality Test (Phase B Gate)

**Question this notebook answers:** can a SAM3-style promptable segmentation model
localize our near-field parking obstacles well enough to auto-generate YOLO training
labels? (Plan: `docs/low_object_detection_plan.md`, Phase B.)

**Protocol:** pick ~2 sequences per target object (spanning low: curbstone, speedbump,
woodenboard — and high: pole, dummychild, bicyclestand), extract frames along the
approach, prompt SAM3 with a text prompt derived from the object name, and inspect the
masks. The output is overlay images + a detection-rate table per object.

**Pass/fail reading:**
- Boxes, poles, dummies found reliably at 2-4 m -> Phase B is viable; the remaining work
  is filtering (corridor, temporal consistency, USS height check).
- Curbs/speedbumps ragged or missed -> expected; decide whether to accept lower quality
  for those classes or invest in edge-aware prompting.
- Everything missed -> the auto-label pipeline needs a different labeler; stop before
  building more.

**Requires a GPU runtime** (SAM3 is ~850M params, ~3.4 GB download).

## 1. Setup

In [ ]:
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

if IN_COLAB:
    # SAM3 needs a recent transformers release.
    pip_install("-U", "transformers", "accelerate")

import json
import random
import re
from collections import defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"colab={IN_COLAB} device={DEVICE}")
if DEVICE != "cuda":
    print("WARNING: SAM3 on CPU will be extremely slow - use a GPU runtime")

## 2. Config

Sequence selection reads the **inventory output** from
`notebooks/build_object_inventory.ipynb` (`inventory_folders.csv`). Either let the
notebook auto-pick `N_PER_OBJECT` random complete sequences per target object, or set
`MANUAL_SEQUENCE_IDS` to pin exact folders.

In [ ]:
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

def find_repo_root() -> Path | None:
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "pyproject.toml").exists():
            return base
    colab_clone = Path("/content/vision-uss-research")
    return colab_clone if colab_clone.exists() else None

PROFILE = "colab_drive" if IN_COLAB else "local"
RAW_ROOT_OVERRIDE = None
INVENTORY_CSV_OVERRIDE = None   # path to inventory_folders.csv
OUTPUT_DIR_OVERRIDE = None

repo_root = find_repo_root()
if repo_root:
    sys.path.insert(0, str(repo_root / "src"))
    from vision_uss_research.settings import load_paths

    paths = load_paths(profile=PROFILE)
    RAW_ROOT = Path(RAW_ROOT_OVERRIDE or paths.raw_data_root)
    INVENTORY_CSV = Path(
        INVENTORY_CSV_OVERRIDE
        or paths.outputs_dir / "profiles" / "object_inventory" / "inventory_folders.csv"
    )
    OUTPUT_DIR = Path(OUTPUT_DIR_OVERRIDE or paths.outputs_dir / "experiments" / "sam3_zero_shot")
else:
    RAW_ROOT = Path(RAW_ROOT_OVERRIDE)
    INVENTORY_CSV = Path(INVENTORY_CSV_OVERRIDE)
    OUTPUT_DIR = Path(OUTPUT_DIR_OVERRIDE)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- experiment knobs -------------------------------------------------------
TARGET_OBJECTS = ["curbstone", "speedbump", "woodenboard",   # low
                  "pole", "dummychild", "bicyclestand"]      # high
N_PER_OBJECT = 2
MANUAL_SEQUENCE_IDS: list[str] = []   # non-empty -> overrides auto-selection
FRAME_RATIOS = [0.40, 0.60, 0.80]     # mid-approach: at ~0.95 the object drops below
                                      # the bumper FOV and detection collapses (v1 run)
SCORE_THRESHOLD = 0.3                 # dummychild/bicyclestand score 0.42-0.57 when
                                      # found; keep low and filter downstream
SEED = 0
SAM3_MODEL_ID = "facebook/sam3"

print(f"raw root  : {RAW_ROOT}")
print(f"inventory : {INVENTORY_CSV}")
print(f"outputs   : {OUTPUT_DIR}")

In [ ]:
# Text prompts per canonical object name. SAM3 only needs to LOCALIZE - the class
# comes from the sequence metadata, so prompts should describe appearance.
OBJECT_PROMPTS = {
    "curbstone": "curb",
    "curbstone_side": "curb",
    "speedbump": "speed bump",
    "woodenboard": "wooden plank lying on the ground",
    "hose": "hose lying on the ground",
    "step": "step edge",
    "pole": "vertical pole",
    "squarepole": "vertical pole",
    "dummychild": "child mannequin",
    "bicyclestand": "bicycle rack",   # v1 prompt 'metal bicycle stand' scored 0.47 max
    "cone": "traffic cone",
    "bollard": "bollard",
    "bush": "bush",
    "tree": "tree trunk",
    "car": "car",
    "stonelarge": "large stone block",
    "stonemiddle": "stone block",
    "cubestandard": "cube-shaped test object",
    "ubarrier": "metal barrier",
    "fence": "fence",
}

def prompt_for(object_name: str) -> str:
    key = re.sub(r"[^a-z0-9_]+", "", object_name.lower())
    if key in OBJECT_PROMPTS:
        return OBJECT_PROMPTS[key]
    # placement variants: 'Dummychild -Central' -> 'dummychild', 'Pole - Displaced' -> 'pole'
    for base, prompt in OBJECT_PROMPTS.items():
        if key.startswith(base):
            return prompt
    return object_name.replace("_", " ")

## 3. Select test sequences from the inventory

In [ ]:
inv = pd.read_csv(INVENTORY_CSV)
usable = inv[
    (inv["status"] == "complete")
    & ((inv["has_front_video"] == 1) | (inv["has_rear_video"] == 1))
].copy()

rng = random.Random(SEED)
selected: list[dict] = []

if MANUAL_SEQUENCE_IDS:
    for seq_id in MANUAL_SEQUENCE_IDS:
        row = inv[inv["sequence_id"] == seq_id]
        assert len(row), f"sequence not in inventory: {seq_id}"
        objs = str(row.iloc[0]["label_v2_objects"]).split("|")
        target = next((o for o in objs if o.lower() != "empty"), objs[0])
        selected.append({"sequence_id": seq_id, "target_object": target})
else:
    for obj in TARGET_OBJECTS:
        # sequence must contain the object in its Label V2 rows
        mask = usable["label_v2_objects"].fillna("").str.split("|").apply(
            lambda names: any(n.strip().lower() == obj for n in names)
        )
        candidates = usable[mask]["sequence_id"].tolist()
        picks = rng.sample(candidates, min(N_PER_OBJECT, len(candidates)))
        selected += [{"sequence_id": s, "target_object": obj} for s in picks]
        print(f"{obj:<15} {len(candidates):>5} candidates -> picked {len(picks)}")

selected_df = pd.DataFrame(selected)
selected_df["prompt"] = selected_df["target_object"].map(prompt_for)
display(selected_df)

## 4. Sequence helpers (camera selection, frame extraction)

In [ ]:
VIDEO_EXTENSIONS = {".mp4", ".webm", ".avi"}

def sequence_files(seq_dir: Path) -> dict:
    top = [p for p in seq_dir.iterdir() if p.is_file()]
    def pick(pred):
        matches = sorted(p for p in top if pred(p))
        return matches[0] if matches else None
    return {
        "metadata_json": pick(lambda p: p.suffix.lower() == ".json"),
        "front_video": pick(lambda p: "front" in p.name.lower() and p.suffix.lower() in VIDEO_EXTENSIONS),
        "rear_video": pick(lambda p: "rear" in p.name.lower() and p.suffix.lower() in VIDEO_EXTENSIONS),
    }

def select_camera_videos(seq_dir: Path) -> tuple[list[tuple[str, Path]], str]:
    """Camera videos to evaluate. When driving direction is unknown, return BOTH
    cameras - guessing wrong shows SAM3 frames where the object never appears
    (this inflated woodenboard/dummychild failures in the v1 run)."""
    found = sequence_files(seq_dir)
    direction = "unknown"
    if found["metadata_json"]:
        with found["metadata_json"].open() as f:
            metadata = json.load(f)
        for tag in metadata.get("tags", []):
            if tag.get("tagSubCategory", {}).get("name", "").lower() == "driving direction":
                direction = tag.get("name", "unknown")

    if direction == "backward":
        videos = [("rear", found["rear_video"])]
    elif direction == "forward":
        videos = [("front", found["front_video"])]
    else:
        videos = [("front", found["front_video"]), ("rear", found["rear_video"])]

    videos = [(name, p) for name, p in videos if p is not None]
    assert videos, f"no camera video in {seq_dir}"
    return videos, direction

def extract_frames(video_path: Path, ratios: list[float]) -> dict[float, np.ndarray]:
    cap = cv2.VideoCapture(str(video_path))
    duration_ms = 1000.0 * cap.get(cv2.CAP_PROP_FRAME_COUNT) / max(cap.get(cv2.CAP_PROP_FPS), 1e-6)
    frames = {}
    for r in ratios:
        cap.set(cv2.CAP_PROP_POS_MSEC, r * duration_ms)
        ok, frame_bgr = cap.read()
        if ok:
            frames[r] = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    cap.release()
    return frames

## 5. Load SAM3

Text-prompted ("promptable concept segmentation") interface via `transformers`.
If your installed version exposes a different API, check the model card at
`https://huggingface.co/facebook/sam3` and adjust the two marked lines.

In [ ]:
from transformers import Sam3Model, Sam3Processor

processor = Sam3Processor.from_pretrained(SAM3_MODEL_ID)
model = Sam3Model.from_pretrained(SAM3_MODEL_ID).to(DEVICE).eval()

@torch.no_grad()
def sam3_segment(image_rgb: np.ndarray, text_prompt: str) -> dict:
    """Returns dict with 'masks' (N,H,W bool), 'boxes' (N,4 xyxy), 'scores' (N,)."""
    pil = Image.fromarray(image_rgb)
    inputs = processor(images=pil, text=text_prompt, return_tensors="pt").to(DEVICE)  # API line 1
    with torch.autocast(DEVICE, enabled=(DEVICE == "cuda")):
        outputs = model(**inputs)
    results = processor.post_process_instance_segmentation(                            # API line 2
        outputs, threshold=SCORE_THRESHOLD, mask_threshold=0.5,
        target_sizes=[(pil.height, pil.width)],
    )[0]
    masks = results["masks"].cpu().numpy().astype(bool)
    boxes = results["boxes"].cpu().numpy()
    scores = results["scores"].cpu().numpy()
    order = np.argsort(-scores)
    return {"masks": masks[order], "boxes": boxes[order], "scores": scores[order]}

## 6. Run over the selected sequences

In [ ]:
PALETTE = [(255, 60, 60), (60, 200, 255), (80, 255, 120), (255, 200, 40), (200, 100, 255)]

def draw_overlay(image_rgb: np.ndarray, result: dict, title: str) -> np.ndarray:
    canvas = image_rgb.copy()
    for i, (mask, box, score) in enumerate(zip(result["masks"], result["boxes"], result["scores"])):
        color = np.array(PALETTE[i % len(PALETTE)], dtype=np.uint8)
        canvas[mask] = (0.55 * canvas[mask] + 0.45 * color).astype(np.uint8)
        x0, y0, x1, y1 = box.astype(int)
        cv2.rectangle(canvas, (x0, y0), (x1, y1), color.tolist(), 2)
        cv2.putText(canvas, f"{score:.2f}", (x0, max(y0 - 6, 12)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.55, color.tolist(), 2)
    cv2.putText(canvas, title, (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
    return canvas

records = []
overlays: dict[str, list[np.ndarray]] = defaultdict(list)

for _, sel in selected_df.iterrows():
    seq_id, target, prompt = sel["sequence_id"], sel["target_object"], sel["prompt"]
    seq_dir = RAW_ROOT / seq_id
    try:
        videos, direction = select_camera_videos(seq_dir)
    except Exception as e:
        print(f"SKIP {seq_id}: {type(e).__name__}: {e}")
        continue

    seq_out = OUTPUT_DIR / seq_id
    seq_out.mkdir(parents=True, exist_ok=True)

    for camera, video_path in videos:
        frames = extract_frames(video_path, FRAME_RATIOS)
        for r, img in frames.items():
            result = sam3_segment(img, prompt)
            n = len(result["scores"])
            top = float(result["scores"][0]) if n else 0.0
            area = float(result["masks"][0].mean()) if n else 0.0

            overlay = draw_overlay(img, result, f"{target} | {camera} | '{prompt}' | r={r}")
            cv2.imwrite(str(seq_out / f"overlay_{camera}_r{int(r * 100):02d}.jpg"),
                        cv2.cvtColor(overlay, cv2.COLOR_RGB2BGR))
            overlays[target].append(overlay)

            records.append({
                "sequence_id": seq_id, "target_object": target, "prompt": prompt,
                "driving_direction": direction, "camera": camera, "frame_ratio": r,
                "n_masks": n, "top_score": round(top, 3),
                "top_mask_area_frac": round(area, 4),
            })
    print(f"done {seq_id} ({target}, cameras={[c for c, _ in videos]})")

results_df = pd.DataFrame(records)
results_df.to_csv(OUTPUT_DIR / "sam3_zero_shot_results.csv", index=False)
print(f"\n{len(results_df)} frame results -> {OUTPUT_DIR / 'sam3_zero_shot_results.csv'}")

## 7. Inspect: overlays per object

In [ ]:
for target, imgs in overlays.items():
    n = len(imgs)
    cols = min(3, n)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, img in zip(axes.flat, imgs):
        ax.imshow(img)
    fig.suptitle(f"SAM3 zero-shot: {target}", fontsize=13)
    plt.show()

## 8. Detection-rate summary (the gate metric)

In [ ]:
# Best-over-cameras per sequence x frame ratio: when direction was unknown both
# cameras were run, but only one can actually show the object.
per_frame = (
    results_df.groupby(["target_object", "sequence_id", "frame_ratio"])
    .agg(detected=("n_masks", lambda s: bool((s > 0).any())),
         top_score=("top_score", "max"),
         n_masks=("n_masks", "max"))
    .reset_index()
)
summary = (
    per_frame.groupby("target_object")
    .agg(
        frames=("detected", "size"),
        detection_rate=("detected", "mean"),
        mean_top_score=("top_score", "mean"),
        mean_n_masks=("n_masks", "mean"),
    )
    .round(3)
    .sort_values("detection_rate", ascending=False)
)
display(summary)
summary.to_csv(OUTPUT_DIR / "sam3_detection_rate_by_object.csv")

print("""
Reading the gate:
- detection_rate is NECESSARY not sufficient - open the overlays and check the masks
  actually cover the target (not a background look-alike).
- mean_n_masks >> 1 means distractors: the filtering stage (approach corridor, temporal
  consistency, USS height check) has real work to do.
- If low objects (curbstone/speedbump/woodenboard) fail while high objects pass, decide:
  accept lower label quality for low classes, or add edge-aware prompting for curbs.
""")

## Next steps after the gate

- **Pass:** build the Phase B batch labeler (script, not notebook): all low/high
  sequences, corridor + temporal + USS-height filtering, YOLO-format export, and the
  mandatory ~200-frame human QA sample (`docs/low_object_detection_plan.md`).
- **Partial:** rerun with more `N_PER_OBJECT` and tuned prompts for the failing classes
  before deciding.
- **Fail:** try grounding-DINO-style detectors or SAM3 with box prompts from the USS
  corridor before abandoning auto-labeling.